<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Bayesian-Embeddings/blob/main/Sim1_Bayesian_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# SIMULATION 1
# Structural recovery + posterior calibration + adaptive budget
# Fixed-dimensional spectral shrinkage + blocked MCMC
#
# Google Colab: ONE CELL
#
# At startup:
#   1 -> start from beginning
#   2 -> resume from latest Google Drive checkpoint
#
# Checkpoints:
#   - after every completed Monte Carlo replication
#   - additionally every 20 minutes during long computations
# ============================================================

# ---------------------------
# 0. START / RESUME QUESTION
# ---------------------------
MODE = input(
    "Simulation 1: type 1 to START FROM THE BEGINNING "
    "or 2 to RESUME FROM THE LAST CHECKPOINT: "
).strip()

if MODE not in {"1", "2"}:
    raise ValueError("Please run the cell again and enter either 1 or 2.")

# ---------------------------
# 1. IMPORTS
# ---------------------------
import os
import gc
import json
import math
import time
import pickle
import random
import hashlib
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# ---------------------------
# 2. GOOGLE DRIVE
# ---------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/JRSSB_Embedding_Allocation/Simulation1"
except Exception:
    ROOT = "./JRSSB_Embedding_Allocation/Simulation1"
    print("Google Drive was not detected. Using local directory:", ROOT)

os.makedirs(ROOT, exist_ok=True)

CHECKPOINT_PATH = os.path.join(ROOT, "simulation1_checkpoint.pkl")
RESULTS_PATH = os.path.join(ROOT, "simulation1_replication_results.csv")
BUDGET_RESULTS_PATH = os.path.join(ROOT, "simulation1_budget_results.csv")
SUMMARY_PATH = os.path.join(ROOT, "simulation1_summary.csv")

# ---------------------------
# 3. CONFIGURATION
# ---------------------------

PILOT_MODE = True

BASE_SEED = 271828
CHECKPOINT_SECONDS = 20 * 60

N_LEVELS = [4, 5, 6, 5]
M = len(N_LEVELS)

P_Z = 4
P_U = 2
Q = 1

K_LIST = [min(nj - 1, P_Z) for nj in N_LEVELS]

N_VAL = 250
N_TEST = 1500

ALPHA = 0.05
ZETA_N = 0.010

THETA_LAMBDA0 = 0.15
TAU_GAMMA = 2.0
TAU_MU = 2.0
SIGMA_A0 = 2.5
SIGMA_B0 = 0.05

REF_HIDDEN = (24, 12)
RED_HIDDEN = (24, 12)

if PILOT_MODE:
    N_REPS = 10

    REF_EPOCHS = 240
    RED_EPOCHS = 200

    MCMC_ITERS = 600
    MCMC_BURN = 300
    MCMC_THIN = 3

else:
    # Recommended once the simulation design is frozen.
    N_REPS = 50

    REF_EPOCHS = 350
    RED_EPOCHS = 280

    MCMC_ITERS = 1200
    MCMC_BURN = 600
    MCMC_THIN = 4

SCENARIOS = [
    {
        "name": "baseline",
        "n_train": 400,
        "noise_sd": 0.20,
        "spectral_regime": "clear"
    },
    {
        "name": "small_sample",
        "n_train": 200,
        "noise_sd": 0.20,
        "spectral_regime": "clear"
    },
    {
        "name": "large_sample",
        "n_train": 800,
        "noise_sd": 0.20,
        "spectral_regime": "clear"
    },
    {
        "name": "high_noise",
        "n_train": 400,
        "noise_sd": 0.35,
        "spectral_regime": "clear"
    },
    {
        "name": "weak_spectral_separation",
        "n_train": 400,
        "noise_sd": 0.20,
        "spectral_regime": "weak"
    },
    {
        "name": "difficult",
        "n_train": 200,
        "noise_sd": 0.35,
        "spectral_regime": "weak"
    },
]

MIN_COST = sum(N_LEVELS)
MAX_COST = sum(
    N_LEVELS[j] * K_LIST[j]
    for j in range(M)
)

BUDGETS = sorted(
    set(
        int(round(x))
        for x in np.linspace(MIN_COST, MAX_COST, 6)
    )
)

BUDGETS[0] = MIN_COST
BUDGETS[-1] = MAX_COST
BUDGETS = sorted(set(BUDGETS))

# ---------------------------
# 4. CONFIG SIGNATURE
# ---------------------------

CONFIG_FOR_HASH = {
    "pilot": PILOT_MODE,
    "n_reps": N_REPS,
    "n_levels": N_LEVELS,
    "p_z": P_Z,
    "p_u": P_U,
    "n_val": N_VAL,
    "n_test": N_TEST,
    "budgets": BUDGETS,
    "zeta": ZETA_N,
    "theta_lambda0": THETA_LAMBDA0,
    "ref_epochs": REF_EPOCHS,
    "red_epochs": RED_EPOCHS,
    "mcmc_iters": MCMC_ITERS,
    "mcmc_burn": MCMC_BURN,
    "mcmc_thin": MCMC_THIN,
    "scenarios": SCENARIOS,
}

CONFIG_SIGNATURE = hashlib.md5(
    json.dumps(CONFIG_FOR_HASH, sort_keys=True).encode()
).hexdigest()

# ---------------------------
# 5. REPRODUCIBILITY
# ---------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ---------------------------
# 6. TRUE EMBEDDING MATRICES
# ---------------------------

def centered_orthonormal_matrix(n, k, rng):
    """
    Generates U in R^{n x k} satisfying:
        U^T U = I
        1^T U = 0
    """
    A = rng.normal(size=(n, k))
    A = A - A.mean(axis=0, keepdims=True)

    Qm, _ = np.linalg.qr(A)

    # Numerical recentering.
    Qm = Qm - Qm.mean(axis=0, keepdims=True)
    Qm, _ = np.linalg.qr(Qm)

    return Qm[:, :k]


def make_theta_from_singular_values(n, pz, singular_values, rng):
    k = min(n - 1, pz)

    U = centered_orthonormal_matrix(n, k, rng)

    V0 = rng.normal(size=(pz, k))
    V, _ = np.linalg.qr(V0)
    V = V[:, :k]

    s = np.asarray(singular_values[:k], dtype=float)

    theta = U @ np.diag(s) @ V.T
    theta = theta - theta.mean(axis=0, keepdims=True)

    return theta


def get_spectral_values(regime):
    if regime == "clear":
        base = [
            [1.60, 0.55, 0.12, 0.03],
            [1.50, 0.90, 0.25, 0.05],
            [1.70, 1.00, 0.45, 0.10],
            [1.40, 0.65, 0.20, 0.04],
        ]

    elif regime == "weak":
        base = [
            [1.60, 1.05, 0.75, 0.55],
            [1.50, 1.15, 0.85, 0.60],
            [1.70, 1.20, 0.90, 0.65],
            [1.40, 1.00, 0.75, 0.55],
        ]

    else:
        raise ValueError("Unknown spectral regime.")

    return [
        np.asarray(base[j][:K_LIST[j]], dtype=float)
        for j in range(M)
    ]


# ---------------------------
# 7. TEACHER MLP
# ---------------------------

def make_teacher_weights(pz, seed=8877):
    rng = np.random.default_rng(seed)

    h1 = REF_HIDDEN[0]
    h2 = REF_HIDDEN[1]

    W1 = rng.normal(
        scale=0.65 / np.sqrt(pz),
        size=(h1, pz)
    )
    b1 = rng.normal(scale=0.10, size=h1)

    W2 = rng.normal(
        scale=0.65 / np.sqrt(h1),
        size=(h2, h1)
    )
    b2 = rng.normal(scale=0.10, size=h2)

    W3 = rng.normal(
        scale=0.90 / np.sqrt(h2),
        size=(1, h2)
    )
    b3 = np.array([0.05])

    return [(W1, b1), (W2, b2), (W3, b3)]


TEACHER_WEIGHTS = make_teacher_weights(P_Z)


def mlp_forward_numpy(z, weights):
    h = np.asarray(z, dtype=float)

    for ell, (W, b) in enumerate(weights):
        h = h @ W.T + b

        if ell < len(weights) - 1:
            h = np.tanh(h)

    return h


# ---------------------------
# 8. FIXED TRUE PARAMETERS
# ---------------------------

def build_true_parameters(spectral_regime):
    rng = np.random.default_rng(12345)

    singular_values = get_spectral_values(spectral_regime)

    theta_true = []

    for j in range(M):
        theta_true.append(
            make_theta_from_singular_values(
                N_LEVELS[j],
                P_Z,
                singular_values[j],
                rng
            )
        )

    gamma_true = rng.normal(
        scale=0.30,
        size=(P_U, P_Z)
    )

    mu_true = np.array([0.15, -0.10, 0.08, 0.05])

    return {
        "theta": theta_true,
        "gamma": gamma_true,
        "mu": mu_true,
        "singular_values": singular_values
    }


# ---------------------------
# 9. DATA GENERATION
# ---------------------------

def generate_dataset(n, true_par, noise_sd, seed):
    rng = np.random.default_rng(seed)

    C = [
        rng.integers(0, N_LEVELS[j], size=n)
        for j in range(M)
    ]

    U = rng.normal(size=(n, P_U))

    z = np.tile(true_par["mu"], (n, 1))
    z = z + U @ true_par["gamma"]

    for j in range(M):
        z = z + true_par["theta"][j][C[j], :]

    y_mean = mlp_forward_numpy(
        z,
        TEACHER_WEIGHTS
    )

    y = (
        y_mean
        + noise_sd * rng.normal(size=y_mean.shape)
    )

    return C, U, y, y_mean


# ---------------------------
# 10. REFERENCE MODEL
# ---------------------------

class ReferenceModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(N_LEVELS[j], P_Z)
                for j in range(M)
            ]
        )

        for emb in self.embeddings:
            nn.init.normal_(emb.weight, std=0.10)

        self.gamma = nn.Parameter(
            0.10 * torch.randn(P_U, P_Z)
        )

        self.mu = nn.Parameter(
            torch.zeros(P_Z)
        )

        self.mlp = nn.Sequential(
            nn.Linear(P_Z, REF_HIDDEN[0]),
            nn.Tanh(),
            nn.Linear(REF_HIDDEN[0], REF_HIDDEN[1]),
            nn.Tanh(),
            nn.Linear(REF_HIDDEN[1], Q)
        )

    def latent(self, C, U):
        z = self.mu.unsqueeze(0) + U @ self.gamma

        for j in range(M):
            z = z + self.embeddings[j](C[j])

        return z

    def forward(self, C, U):
        return self.mlp(
            self.latent(C, U)
        )

    @torch.no_grad()
    def center_embeddings_(self):
        for emb in self.embeddings:
            emb.weight -= emb.weight.mean(
                dim=0,
                keepdim=True
            )


def train_reference_model(C, U, y, seed):
    set_seed(seed)

    model = ReferenceModel().to(DEVICE)

    C_t = [
        torch.tensor(x, dtype=torch.long, device=DEVICE)
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.010,
        weight_decay=1e-5
    )

    for epoch in range(REF_EPOCHS):
        optimizer.zero_grad()

        pred = model(C_t, U_t)

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        optimizer.step()
        model.center_embeddings_()

    return model


@torch.no_grad()
def predict_reference(model, C, U):
    C_t = [
        torch.tensor(x, dtype=torch.long, device=DEVICE)
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(C_t, U_t)
        .cpu()
        .numpy()
    )


def extract_reference_parameters(model):
    theta = [
        emb.weight.detach().cpu().numpy().astype(float).copy()
        for emb in model.embeddings
    ]

    theta = [
        x - x.mean(axis=0, keepdims=True)
        for x in theta
    ]

    gamma = (
        model.gamma
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    mu = (
        model.mu
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    weights = []

    for layer in model.mlp:
        if isinstance(layer, nn.Linear):
            W = (
                layer.weight
                .detach()
                .cpu()
                .numpy()
                .astype(float)
                .copy()
            )

            b = (
                layer.bias
                .detach()
                .cpu()
                .numpy()
                .astype(float)
                .copy()
            )

            weights.append((W, b))

    return theta, gamma, mu, weights


# ---------------------------
# 11. SPECTRAL SHRINKAGE PRIOR
# ---------------------------

def log_theta_prior(theta, lambda0=THETA_LAMBDA0):
    k = min(
        theta.shape[0] - 1,
        theta.shape[1]
    )

    s = np.linalg.svd(
        theta,
        compute_uv=False
    )[:k]

    lam = (
        lambda0
        * np.arange(1, k + 1, dtype=float) ** 2
    )

    return -0.5 * np.sum(
        lam * s ** 2
    )


def log_gamma_mu_prior(gamma, mu):
    return (
        -0.5 * np.sum(gamma ** 2) / TAU_GAMMA ** 2
        -0.5 * np.sum(mu ** 2) / TAU_MU ** 2
    )


# ---------------------------
# 12. LIKELIHOOD
# ---------------------------

def gaussian_loglik(y, yhat, sigma2):
    n = y.size

    resid = y - yhat

    return (
        -0.5 * n * np.log(
            2.0 * np.pi * sigma2
        )
        -0.5 * np.sum(resid ** 2) / sigma2
    )


def latent_numpy(C, U, theta, gamma, mu):
    n = len(C[0])

    z = np.tile(mu, (n, 1))
    z += U @ gamma

    for j in range(len(theta)):
        z += theta[j][C[j], :]

    return z


# ---------------------------
# 13. BLOCKED MCMC
# ---------------------------

def blocked_mcmc(
    C,
    U,
    y,
    theta_init,
    gamma_init,
    mu_init,
    fixed_mlp_weights,
    sigma2_init,
    seed,
    heartbeat=None
):
    rng = np.random.default_rng(seed)

    theta = [
        x.copy()
        for x in theta_init
    ]

    gamma = gamma_init.copy()
    mu = mu_init.copy()

    sigma2 = float(
        max(sigma2_init, 1e-5)
    )

    z_cur = latent_numpy(
        C,
        U,
        theta,
        gamma,
        mu
    )

    yhat_cur = mlp_forward_numpy(
        z_cur,
        fixed_mlp_weights
    )

    ll_cur = gaussian_loglik(
        y,
        yhat_cur,
        sigma2
    )

    theta_prior_cur = [
        log_theta_prior(x)
        for x in theta
    ]

    gm_prior_cur = log_gamma_mu_prior(
        gamma,
        mu
    )

    theta_steps = np.full(
        M,
        0.025,
        dtype=float
    )

    gm_step = 0.020

    theta_accept = np.zeros(M)
    theta_trials = np.zeros(M)

    theta_window_accept = np.zeros(M)
    theta_window_trials = np.zeros(M)

    gm_accept = 0
    gm_trials = 0

    gm_window_accept = 0
    gm_window_trials = 0

    singular_draws = [
        []
        for _ in range(M)
    ]

    sigma_draws = []

    adapt_every = 50
    target_acceptance = 0.25

    for it in range(MCMC_ITERS):

        # ---------------------------------
        # Predictor-specific Theta updates
        # ---------------------------------
        for j in range(M):

            proposal = (
                theta[j]
                + theta_steps[j]
                * rng.normal(
                    size=theta[j].shape
                )
            )

            proposal -= proposal.mean(
                axis=0,
                keepdims=True
            )

            delta_latent = (
                proposal[C[j], :]
                - theta[j][C[j], :]
            )

            z_prop = z_cur + delta_latent

            yhat_prop = mlp_forward_numpy(
                z_prop,
                fixed_mlp_weights
            )

            ll_prop = gaussian_loglik(
                y,
                yhat_prop,
                sigma2
            )

            prior_prop = log_theta_prior(
                proposal
            )

            log_ratio = (
                ll_prop
                + prior_prop
                - ll_cur
                - theta_prior_cur[j]
            )

            theta_trials[j] += 1
            theta_window_trials[j] += 1

            if np.log(rng.uniform()) < log_ratio:
                theta[j] = proposal
                z_cur = z_prop
                yhat_cur = yhat_prop
                ll_cur = ll_prop
                theta_prior_cur[j] = prior_prop

                theta_accept[j] += 1
                theta_window_accept[j] += 1

        # ---------------------------------
        # Joint Gamma + mu update
        # ---------------------------------
        gamma_prop = (
            gamma
            + gm_step
            * rng.normal(size=gamma.shape)
        )

        mu_prop = (
            mu
            + gm_step
            * rng.normal(size=mu.shape)
        )

        z_prop = (
            z_cur
            + U @ (gamma_prop - gamma)
            + (mu_prop - mu)
        )

        yhat_prop = mlp_forward_numpy(
            z_prop,
            fixed_mlp_weights
        )

        ll_prop = gaussian_loglik(
            y,
            yhat_prop,
            sigma2
        )

        gm_prior_prop = log_gamma_mu_prior(
            gamma_prop,
            mu_prop
        )

        log_ratio = (
            ll_prop
            + gm_prior_prop
            - ll_cur
            - gm_prior_cur
        )

        gm_trials += 1
        gm_window_trials += 1

        if np.log(rng.uniform()) < log_ratio:
            gamma = gamma_prop
            mu = mu_prop

            z_cur = z_prop
            yhat_cur = yhat_prop

            ll_cur = ll_prop
            gm_prior_cur = gm_prior_prop

            gm_accept += 1
            gm_window_accept += 1

        # ---------------------------------
        # Gibbs update for sigma^2
        # ---------------------------------
        resid = y - yhat_cur

        shape_post = (
            SIGMA_A0
            + 0.5 * y.size
        )

        rate_post = (
            SIGMA_B0
            + 0.5 * np.sum(resid ** 2)
        )

        sigma2 = 1.0 / rng.gamma(
            shape=shape_post,
            scale=1.0 / rate_post
        )

        ll_cur = gaussian_loglik(
            y,
            yhat_cur,
            sigma2
        )

        # ---------------------------------
        # Adapt proposal scales in burn-in
        # ---------------------------------
        if (
            it < MCMC_BURN
            and (it + 1) % adapt_every == 0
        ):

            for j in range(M):

                if theta_window_trials[j] > 0:

                    rate = (
                        theta_window_accept[j]
                        / theta_window_trials[j]
                    )

                    theta_steps[j] *= np.exp(
                        0.50
                        * (rate - target_acceptance)
                    )

                    theta_steps[j] = np.clip(
                        theta_steps[j],
                        0.001,
                        0.15
                    )

            if gm_window_trials > 0:

                rate = (
                    gm_window_accept
                    / gm_window_trials
                )

                gm_step *= np.exp(
                    0.50
                    * (rate - target_acceptance)
                )

                gm_step = float(
                    np.clip(
                        gm_step,
                        0.001,
                        0.15
                    )
                )

            theta_window_accept[:] = 0
            theta_window_trials[:] = 0

            gm_window_accept = 0
            gm_window_trials = 0

        # ---------------------------------
        # Save posterior draw
        # ---------------------------------
        if (
            it >= MCMC_BURN
            and
            (it - MCMC_BURN) % MCMC_THIN == 0
        ):

            for j in range(M):

                s = np.linalg.svd(
                    theta[j],
                    compute_uv=False
                )[:K_LIST[j]]

                singular_draws[j].append(s)

            sigma_draws.append(sigma2)

        if (
            heartbeat is not None
            and (it + 1) % 50 == 0
        ):
            heartbeat()

    singular_draws = [
        np.asarray(x)
        for x in singular_draws
    ]

    mean_s2 = [
        np.mean(x ** 2, axis=0)
        for x in singular_draws
    ]

    return {
        "singular_draws": singular_draws,
        "mean_s2": mean_s2,
        "sigma_draws": np.asarray(sigma_draws),
        "theta_acceptance": (
            theta_accept
            / np.maximum(theta_trials, 1)
        ),
        "gm_acceptance": (
            gm_accept
            / max(gm_trials, 1)
        ),
        "theta_steps": theta_steps,
        "gm_step": gm_step,
    }


# ---------------------------
# 14. STRUCTURAL LOSS
# ---------------------------

def tail_loss_arrays(mean_s2):
    losses = []

    for s2 in mean_s2:
        k = len(s2)

        loss_j = np.array(
            [
                np.sum(s2[d:])
                for d in range(1, k + 1)
            ],
            dtype=float
        )

        losses.append(loss_j)

    return losses


def allocation_loss(d, loss_arrays):
    return float(
        sum(
            loss_arrays[j][d[j] - 1]
            for j in range(len(d))
        )
    )


def embedding_cost(d, n_levels=N_LEVELS):
    return int(
        sum(
            n_levels[j] * d[j]
            for j in range(len(d))
        )
    )


# ---------------------------
# 15. MULTIPLE-CHOICE KNAPSACK DP
# ---------------------------

def dp_allocations(
    n_levels,
    k_list,
    loss_arrays,
    budgets
):
    m = len(n_levels)
    bmax = max(budgets)

    inf = np.inf

    dp = np.full(
        (m + 1, bmax + 1),
        inf
    )

    parent_d = np.full(
        (m + 1, bmax + 1),
        -1,
        dtype=int
    )

    parent_b = np.full(
        (m + 1, bmax + 1),
        -1,
        dtype=int
    )

    dp[0, 0] = 0.0

    for j in range(1, m + 1):

        Nj = n_levels[j - 1]
        Kj = k_list[j - 1]

        for b in range(bmax + 1):

            for d in range(1, Kj + 1):

                cost = Nj * d

                if (
                    b >= cost
                    and np.isfinite(
                        dp[j - 1, b - cost]
                    )
                ):

                    value = (
                        dp[j - 1, b - cost]
                        + loss_arrays[j - 1][d - 1]
                    )

                    if value < dp[j, b]:
                        dp[j, b] = value
                        parent_d[j, b] = d
                        parent_b[j, b] = b - cost

    out = {}

    for B in budgets:

        feasible_costs = np.arange(B + 1)

        vals = dp[m, feasible_costs]

        if not np.any(np.isfinite(vals)):
            raise RuntimeError(
                f"No feasible allocation for budget {B}"
            )

        best_cost = int(
            feasible_costs[np.nanargmin(vals)]
        )

        dvec = [None] * m
        b = best_cost

        for j in range(m, 0, -1):

            d = int(parent_d[j, b])

            if d < 1:
                raise RuntimeError(
                    "DP backtracking failed."
                )

            dvec[j - 1] = d
            b = int(parent_b[j, b])

        out[B] = {
            "d": tuple(dvec),
            "loss": float(dp[m, best_cost]),
            "actual_cost": best_cost,
        }

    return out


# ---------------------------
# 16. REDUCED CANDIDATE MODEL
# ---------------------------

class ReducedModel(nn.Module):

    def __init__(self, d):
        super().__init__()

        self.d = list(d)

        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    N_LEVELS[j],
                    self.d[j]
                )
                for j in range(M)
            ]
        )

        for emb in self.embeddings:
            nn.init.normal_(
                emb.weight,
                std=0.10
            )

        self.gamma = nn.Parameter(
            0.10 * torch.randn(P_U, P_Z)
        )

        self.mu = nn.Parameter(
            torch.zeros(P_Z)
        )

        input_dim = (
            sum(self.d)
            + P_Z
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                RED_HIDDEN[0]
            ),
            nn.Tanh(),
            nn.Linear(
                RED_HIDDEN[0],
                RED_HIDDEN[1]
            ),
            nn.Tanh(),
            nn.Linear(
                RED_HIDDEN[1],
                Q
            )
        )

    def latent(self, C, U):

        pieces = [
            self.embeddings[j](C[j])
            for j in range(M)
        ]

        continuous = (
            self.mu.unsqueeze(0)
            + U @ self.gamma
        )

        pieces.append(continuous)

        return torch.cat(
            pieces,
            dim=1
        )

    def forward(self, C, U):
        return self.mlp(
            self.latent(C, U)
        )

    @torch.no_grad()
    def center_embeddings_(self):
        for emb in self.embeddings:
            emb.weight -= emb.weight.mean(
                dim=0,
                keepdim=True
            )


def train_reduced_model(C, U, y, d, seed):
    set_seed(seed)

    model = ReducedModel(d).to(DEVICE)

    C_t = [
        torch.tensor(x, dtype=torch.long, device=DEVICE)
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.010,
        weight_decay=1e-5
    )

    for epoch in range(RED_EPOCHS):

        optimizer.zero_grad()

        pred = model(C_t, U_t)

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        optimizer.step()
        model.center_embeddings_()

    return model


@torch.no_grad()
def predict_reduced(model, C, U):
    C_t = [
        torch.tensor(x, dtype=torch.long, device=DEVICE)
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(C_t, U_t)
        .cpu()
        .numpy()
    )


def mse(y, pred):
    return float(
        np.mean(
            (y - pred) ** 2
        )
    )


# ---------------------------
# 17. POSTERIOR CALIBRATION
# ---------------------------

def posterior_tail_calibration(
    singular_draws,
    true_singular_values
):
    covered = []
    widths = []

    for j in range(M):

        draws = singular_draws[j]
        true_s = np.asarray(
            true_singular_values[j]
        )

        Kj = len(true_s)

        # d = Kj gives identically zero tail,
        # so exclude that trivial case.
        for d in range(1, Kj):

            posterior_tail = np.sum(
                draws[:, d:] ** 2,
                axis=1
            )

            lower, upper = np.quantile(
                posterior_tail,
                [0.025, 0.975]
            )

            true_tail = float(
                np.sum(
                    true_s[d:] ** 2
                )
            )

            covered.append(
                float(
                    lower <= true_tail <= upper
                )
            )

            widths.append(
                upper - lower
            )

    return (
        float(np.mean(covered))
        if covered else np.nan,
        float(np.mean(widths))
        if widths else np.nan
    )


# ---------------------------
# 18. EXACT ENUMERATION FOR DELTA
# ---------------------------

def all_allocations():
    return itertools.product(
        *[
            range(1, K_LIST[j] + 1)
            for j in range(M)
        ]
    )


def compute_delta_max(
    posterior_losses,
    true_losses
):
    delta = 0.0

    for d in all_allocations():

        diff = abs(
            allocation_loss(
                d,
                posterior_losses
            )
            -
            allocation_loss(
                d,
                true_losses
            )
        )

        delta = max(
            delta,
            diff
        )

    return float(delta)


# ---------------------------
# 19. CHECKPOINT STATE
# ---------------------------

if MODE == "2" and os.path.exists(CHECKPOINT_PATH):

    with open(CHECKPOINT_PATH, "rb") as f:
        STATE = pickle.load(f)

    if (
        STATE.get("config_signature")
        != CONFIG_SIGNATURE
    ):
        raise RuntimeError(
            "The saved checkpoint was created with a "
            "different configuration. Start from the beginning "
            "or restore the previous configuration."
        )

    print(
        "Resumed checkpoint with",
        len(STATE["completed"]),
        "completed replications."
    )

else:

    STATE = {
        "config_signature": CONFIG_SIGNATURE,
        "completed": set(),
        "replication_rows": [],
        "budget_rows": [],
        "started": time.time(),
    }

    if MODE == "2":
        print(
            "No previous checkpoint was found. "
            "Starting from the beginning."
        )


CHECKPOINT_CLOCK = {
    "last": time.time()
}


def save_checkpoint(force=False):

    now = time.time()

    if (
        force
        or
        now - CHECKPOINT_CLOCK["last"]
        >= CHECKPOINT_SECONDS
    ):

        tmp = CHECKPOINT_PATH + ".tmp"

        with open(tmp, "wb") as f:
            pickle.dump(
                STATE,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        os.replace(
            tmp,
            CHECKPOINT_PATH
        )

        CHECKPOINT_CLOCK["last"] = now

        print(
            f"[checkpoint] saved at "
            f"{time.strftime('%H:%M:%S')}"
        )


# ---------------------------
# 20. ONE MONTE CARLO REPLICATION
# ---------------------------

def run_replication(
    scenario,
    scenario_index,
    rep
):
    t0 = time.time()

    seed = (
        BASE_SEED
        + 100000 * scenario_index
        + rep
    )

    true_par = build_true_parameters(
        scenario["spectral_regime"]
    )

    # Independent train / validation / test samples.
    Ctr, Utr, ytr, _ = generate_dataset(
        scenario["n_train"],
        true_par,
        scenario["noise_sd"],
        seed + 11
    )

    Cva, Uva, yva, _ = generate_dataset(
        N_VAL,
        true_par,
        scenario["noise_sd"],
        seed + 22
    )

    Cte, Ute, yte, _ = generate_dataset(
        N_TEST,
        true_par,
        scenario["noise_sd"],
        seed + 33
    )

    # ---------------------------------
    # Reference full model
    # ---------------------------------
    reference = train_reference_model(
        Ctr,
        Utr,
        ytr,
        seed + 100
    )

    ref_train_pred = predict_reference(
        reference,
        Ctr,
        Utr
    )

    ref_test_pred = predict_reference(
        reference,
        Cte,
        Ute
    )

    reference_test_mse = mse(
        yte,
        ref_test_pred
    )

    theta0, gamma0, mu0, omega_fixed = (
        extract_reference_parameters(
            reference
        )
    )

    sigma2_init = max(
        mse(
            ytr,
            ref_train_pred
        ),
        1e-5
    )

    # ---------------------------------
    # Bayesian fixed-dimensional MCMC
    # ---------------------------------
    posterior = blocked_mcmc(
        Ctr,
        Utr,
        ytr,
        theta0,
        gamma0,
        mu0,
        omega_fixed,
        sigma2_init,
        seed + 200,
        heartbeat=lambda: save_checkpoint(
            force=False
        )
    )

    posterior_losses = tail_loss_arrays(
        posterior["mean_s2"]
    )

    # ---------------------------------
    # True structural benchmark
    # ---------------------------------
    true_s2 = [
        np.asarray(s) ** 2
        for s in true_par["singular_values"]
    ]

    true_losses = tail_loss_arrays(
        true_s2
    )

    # ---------------------------------
    # Posterior structural allocations
    # ---------------------------------
    posterior_dp = dp_allocations(
        N_LEVELS,
        K_LIST,
        posterior_losses,
        BUDGETS
    )

    oracle_dp = dp_allocations(
        N_LEVELS,
        K_LIST,
        true_losses,
        BUDGETS
    )

    # ---------------------------------
    # Posterior tail calibration
    # ---------------------------------
    tail_coverage, tail_ci_width = (
        posterior_tail_calibration(
            posterior["singular_draws"],
            true_par["singular_values"]
        )
    )

    delta_max = compute_delta_max(
        posterior_losses,
        true_losses
    )

    # ---------------------------------
    # Fit only unique DP-selected models
    # ---------------------------------
    unique_allocations = sorted(
        set(
            posterior_dp[B]["d"]
            for B in BUDGETS
        )
    )

    candidate_cache = {}

    for candidate_index, d in enumerate(
        unique_allocations
    ):

        alloc_seed = (
            seed
            + 10000
            + 100 * candidate_index
            + sum(
                (j + 1) * d[j]
                for j in range(M)
            )
        )

        reduced = train_reduced_model(
            Ctr,
            Utr,
            ytr,
            d,
            alloc_seed
        )

        val_pred = predict_reduced(
            reduced,
            Cva,
            Uva
        )

        test_pred = predict_reduced(
            reduced,
            Cte,
            Ute
        )

        candidate_cache[d] = {
            "val_mse": mse(
                yva,
                val_pred
            ),
            "test_mse": mse(
                yte,
                test_pred
            ),
            "total_parameters": int(
                sum(
                    p.numel()
                    for p in reduced.parameters()
                )
            )
        }

        del reduced
        gc.collect()

    # ---------------------------------
    # Budget-level results
    # ---------------------------------
    budget_rows = []

    recovery_indicators = []
    structural_excesses = []

    for B in BUDGETS:

        d_post = posterior_dp[B]["d"]
        d_oracle = oracle_dp[B]["d"]

        candidate = candidate_cache[d_post]

        post_struct_true = allocation_loss(
            d_post,
            true_losses
        )

        oracle_struct_true = allocation_loss(
            d_oracle,
            true_losses
        )

        structural_excess = (
            post_struct_true
            - oracle_struct_true
        )

        exact_recovery = float(
            tuple(d_post)
            == tuple(d_oracle)
        )

        recovery_indicators.append(
            exact_recovery
        )

        structural_excesses.append(
            structural_excess
        )

        budget_rows.append(
            {
                "scenario": scenario["name"],
                "rep": rep,
                "B": B,
                "posterior_d": str(tuple(d_post)),
                "oracle_d": str(tuple(d_oracle)),
                "actual_cost": embedding_cost(d_post),
                "posterior_structural_score": (
                    posterior_dp[B]["loss"]
                ),
                "true_structural_loss": (
                    post_struct_true
                ),
                "oracle_structural_loss": (
                    oracle_struct_true
                ),
                "structural_excess": (
                    structural_excess
                ),
                "exact_allocation_recovery": (
                    exact_recovery
                ),
                "validation_mse": (
                    candidate["val_mse"]
                ),
                "test_mse": (
                    candidate["test_mse"]
                ),
                "candidate_total_parameters": (
                    candidate["total_parameters"]
                ),
            }
        )

    # ---------------------------------
    # Adaptive budget selection
    # ---------------------------------
    BMAX = max(BUDGETS)

    d_bmax = posterior_dp[BMAX]["d"]

    reference_validation_level = (
        candidate_cache[d_bmax]["val_mse"]
    )

    admissible = [
        B
        for B in sorted(BUDGETS)
        if (
            candidate_cache[
                posterior_dp[B]["d"]
            ]["val_mse"]
            <= reference_validation_level
            + ZETA_N
        )
    ]

    B_hat = min(admissible)

    d_hat = posterior_dp[B_hat]["d"]

    final_candidate = candidate_cache[d_hat]
    max_candidate = candidate_cache[d_bmax]

    rich_embedding_parameters = (
        sum(
            N_LEVELS[j] * P_Z
            for j in range(M)
        )
    )

    selected_embedding_parameters = (
        embedding_cost(d_hat)
    )

    embedding_reduction = (
        1.0
        -
        selected_embedding_parameters
        / rich_embedding_parameters
    )

    runtime_sec = (
        time.time()
        - t0
    )

    rep_row = {
        "scenario": scenario["name"],
        "rep": rep,
        "n_train": scenario["n_train"],
        "noise_sd": scenario["noise_sd"],
        "spectral_regime": (
            scenario["spectral_regime"]
        ),
        "selected_budget": B_hat,
        "selected_d": str(tuple(d_hat)),
        "selected_embedding_parameters": (
            selected_embedding_parameters
        ),
        "rich_embedding_parameters": (
            rich_embedding_parameters
        ),
        "embedding_reduction": (
            embedding_reduction
        ),
        "allocation_recovery_rate_across_budgets": (
            float(
                np.mean(
                    recovery_indicators
                )
            )
        ),
        "mean_structural_excess_across_budgets": (
            float(
                np.mean(
                    structural_excesses
                )
            )
        ),
        "max_structural_score_error_delta": (
            delta_max
        ),
        "posterior_tail_95_coverage": (
            tail_coverage
        ),
        "posterior_tail_mean_CI_width": (
            tail_ci_width
        ),
        "final_validation_mse": (
            final_candidate["val_mse"]
        ),
        "final_test_mse": (
            final_candidate["test_mse"]
        ),
        "max_budget_test_mse": (
            max_candidate["test_mse"]
        ),
        "reference_full_model_test_mse": (
            reference_test_mse
        ),
        "test_excess_vs_max_budget": (
            final_candidate["test_mse"]
            - max_candidate["test_mse"]
        ),
        "mean_theta_acceptance": float(
            np.mean(
                posterior["theta_acceptance"]
            )
        ),
        "gamma_mu_acceptance": float(
            posterior["gm_acceptance"]
        ),
        "n_unique_refits": len(
            unique_allocations
        ),
        "runtime_seconds": runtime_sec,
    }

    del reference
    gc.collect()

    return rep_row, budget_rows


# ---------------------------
# 21. RUN SIMULATION
# ---------------------------

print("\nSimulation 1")
print("------------------------------")
print("Budgets:", BUDGETS)
print("K_j:", K_LIST)
print("Monte Carlo repetitions per scenario:", N_REPS)
print("Pilot mode:", PILOT_MODE)
print("------------------------------\n")

try:

    for s_idx, scenario in enumerate(
        SCENARIOS
    ):

        for rep in range(1, N_REPS + 1):

            task_key = (
                scenario["name"],
                rep
            )

            if task_key in STATE["completed"]:
                continue

            print(
                f"\n[{scenario['name']}] "
                f"replication {rep}/{N_REPS}"
            )

            rep_row, budget_rows = (
                run_replication(
                    scenario,
                    s_idx,
                    rep
                )
            )

            STATE[
                "replication_rows"
            ].append(rep_row)

            STATE[
                "budget_rows"
            ].extend(budget_rows)

            STATE[
                "completed"
            ].add(task_key)

            # Save every completed replication.
            save_checkpoint(
                force=True
            )

            print(
                "  selected B =",
                rep_row["selected_budget"],
                "| d =",
                rep_row["selected_d"],
                "| test MSE =",
                round(
                    rep_row["final_test_mse"],
                    5
                ),
                "| allocation recovery =",
                round(
                    rep_row[
                        "allocation_recovery_rate_across_budgets"
                    ],
                    3
                )
            )

except KeyboardInterrupt:

    print(
        "\nInterrupted by user. Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise

except Exception:

    print(
        "\nAn error occurred. Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


# ---------------------------
# 22. SAVE FINAL TABLES
# ---------------------------

rep_df = pd.DataFrame(
    STATE["replication_rows"]
)

budget_df = pd.DataFrame(
    STATE["budget_rows"]
)

rep_df.to_csv(
    RESULTS_PATH,
    index=False
)

budget_df.to_csv(
    BUDGET_RESULTS_PATH,
    index=False
)

summary = (
    rep_df
    .groupby("scenario")
    .agg(
        n_rep=(
            "rep",
            "count"
        ),
        allocation_recovery=(
            "allocation_recovery_rate_across_budgets",
            "mean"
        ),
        structural_excess=(
            "mean_structural_excess_across_budgets",
            "mean"
        ),
        delta_score_error=(
            "max_structural_score_error_delta",
            "mean"
        ),
        posterior_tail_coverage=(
            "posterior_tail_95_coverage",
            "mean"
        ),
        final_test_mse=(
            "final_test_mse",
            "mean"
        ),
        max_budget_test_mse=(
            "max_budget_test_mse",
            "mean"
        ),
        reference_test_mse=(
            "reference_full_model_test_mse",
            "mean"
        ),
        test_excess_vs_max_budget=(
            "test_excess_vs_max_budget",
            "mean"
        ),
        mean_selected_budget=(
            "selected_budget",
            "mean"
        ),
        embedding_reduction=(
            "embedding_reduction",
            "mean"
        ),
        mean_unique_refits=(
            "n_unique_refits",
            "mean"
        ),
        runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),
    )
    .reset_index()
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

# ---------------------------
# 23. SIMPLE FIGURES
# ---------------------------

plt.figure(figsize=(9, 5))
plt.bar(
    summary["scenario"],
    summary["allocation_recovery"]
)
plt.ylabel("Mean allocation recovery rate")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(
        ROOT,
        "sim1_allocation_recovery.png"
    ),
    dpi=180
)
plt.close()

plt.figure(figsize=(9, 5))
x = np.arange(len(summary))
width = 0.35

plt.bar(
    x - width / 2,
    summary["final_test_mse"],
    width,
    label="Adaptive allocation"
)

plt.bar(
    x + width / 2,
    summary["max_budget_test_mse"],
    width,
    label="Maximum budget"
)

plt.xticks(
    x,
    summary["scenario"],
    rotation=30,
    ha="right"
)

plt.ylabel("Test MSE")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_predictive_risk.png"
    ),
    dpi=180
)
plt.close()

plt.figure(figsize=(9, 5))
plt.bar(
    summary["scenario"],
    100.0
    * summary["embedding_reduction"]
)
plt.ylabel("Embedding parameter reduction (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_parameter_reduction.png"
    ),
    dpi=180
)
plt.close()

# ---------------------------
# 24. FINAL CHECKPOINT
# ---------------------------

STATE["finished"] = True
save_checkpoint(force=True)

print("\nSIMULATION 1 COMPLETED.")
print("\nSummary:")
print(summary.to_string(index=False))

print("\nFiles saved in:")
print(ROOT)

Simulation 1: type 1 to START FROM THE BEGINNING or 2 to RESUME FROM THE LAST CHECKPOINT: 2
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Resumed checkpoint with 60 completed replications.

Simulation 1
------------------------------
Budgets: [20, 31, 42, 54, 65, 76]
K_j: [3, 4, 4, 4]
Monte Carlo repetitions per scenario: 10
Pilot mode: True
------------------------------

[checkpoint] saved at 10:16:15

SIMULATION 1 COMPLETED.

Summary:
                scenario  n_rep  allocation_recovery  structural_excess  delta_score_error  posterior_tail_coverage  final_test_mse  max_budget_test_mse  reference_test_mse  test_excess_vs_max_budget  mean_selected_budget  embedding_reduction  mean_unique_refits  runtime_seconds
                baseline     10             0.583333           0.096212           1.036325                 0.645455        0.048335             0.059840            0.054425                  -0.01